In [1]:
!pip install opencv-contrib-python

  Using cached opencv_contrib_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached numpy-2.4.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached opencv_contrib_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl (79.2 MB)
Using cached numpy-2.4.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import cv2
print(cv2.getBuildInformation())


General configuration for OpenCV 4.13.0 =====================================
  Version control:               4.13.0-1-gb4c5ec4042-dirty

  Extra modules:
    Location (extra):            /io/opencv_contrib/modules
    Version control (extra):     4.13.0

  Platform:
    Timestamp:                   2026-02-05T07:45:35Z
    Host:                        Linux 6.8.0-1044-azure x86_64
    CMake:                       4.2.1
    CMake generator:             Unix Makefiles
    CMake build tool:            /bin/gmake
    Configuration:               Release
    Algorithm Hint:              ALGO_HINT_ACCURATE

  CPU/HW features:
    Baseline:                    SSE SSE2 SSE3
      requested:                 SSE3
    Dispatched code generation:  SSE4_1 SSE4_2 AVX FP16 AVX2 AVX512_SKX
      SSE4_1 (17 files):         + SSSE3 SSE4_1
      SSE4_2 (1 files):          + SSSE3 SSE4_1 POPCNT SSE4_2
      AVX (9 files):             + SSSE3 SSE4_1 POPCNT SSE4_2 AVX
      FP16 (0 files):            + S

In [3]:
import cv2
from cv2 import dnn_superres

print(cv2.__version__)
print(dir(dnn_superres))

4.13.0
['DnnSuperResImpl', 'DnnSuperResImpl_create', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_native']


In [5]:
import cv2
from cv2 import dnn_superres


sr = dnn_superres.DnnSuperResImpl_create()


sr.readModel("../models/FSRCNN_x4.pb")
sr.setModel("fsrcnn", 4)

img = cv2.imread("/home/umarly-poeta/projects/ai_enhanced_renderer/notebooks/example_data/example.jpg")

result = sr.upsample(img)

cv2.imwrite("/home/umarly-poeta/projects/ai_enhanced_renderer/notebooks/example_data/upscaled/example_upscaled.jpg", result)

True

In [ ]:
#!/usr/bin/env python3
"""
Tydzień 1 Deliverable — Osoba B
Wczytaj obraz JPG i uruchom FSRCNN przez OpenCV dnn_superres.
Użycie: python week1_fsrcnn_demo.py input.jpg
"""
import sys
import cv2
import numpy as np
from cv2 import dnn_superres
import time
import os

def run_fsrcnn(input_path: str, model_path: str = "models/FSRCNN_x4.pb",
              scale: int = 4, output_path: str = None) -> None:
    """Uruchom FSRCNN na obrazie i wyświetl/zapisz wynik."""

    # 1. Wczytaj obraz
    img = cv2.imread(input_path)
    if img is None:
        print(f"❌ Nie można wczytać: {input_path}")
        sys.exit(1)

    h, w = img.shape[:2]
    print(f"📷 Obraz wejściowy: {w}×{h} px")

    # 2. Utwórz obiekt SR i wczytaj model
    sr = dnn_superres.DnnSuperResImpl_create()
    sr.readModel(model_path)
    sr.setModel("fsrcnn", scale)      # ← musi pasować do nazwy pliku

    # 3. Zmierz czas inferencji
    t0 = time.perf_counter()
    result = sr.upsample(img)
    elapsed_ms = (time.perf_counter() - t0) * 1000

    h2, w2 = result.shape[:2]
    print(f"✅ Wynik SR: {w2}×{h2} px")
    print(f"⏱️  Czas inferencji: {elapsed_ms:.1f} ms")

    # 4. Zapisz wynik
    if output_path is None:
        base, ext = os.path.splitext(input_path)
        output_path = f"{base}_FSRCNN_x{scale}.png"

    cv2.imwrite(output_path, result)
    print(f"💾 Zapisano: {output_path}")

    # 5. Pokaż porównanie (opcjonalnie)
    bicubic = cv2.resize(img, (w * scale, h * scale),
                         interpolation=cv2.INTER_CUBIC)
    comparison = np.hstack([bicubic, result])
    cv2.imshow(f"Bicubic  |  FSRCNN x{scale}", comparison)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

if __name__ == "__main__":
    image_path = sys.argv[1] if len(sys.argv) > 1 else "images/test.jpg"
    run_fsrcnn(image_path)